# Phase 4 — what ships is the average, not the argmax

The run used to end with an *argmax*: fit `lgbm`, `xgb`, `catboost` and
`elasticnet` on the winning table, and ship whichever scored best on the
validation half.

Rossmann v7 was the first run with a sealed third split, and it caught that
ending being wrong:

| model | val | **sealed** | |
|---|---|---|---|
| **xgb** | **0.12980** | **0.15253** | shipped — and the *worst* of the three trees |
| lgbm | 0.13202 | 0.14145 | |
| catboost | 0.14329 | **0.14113** | actually the best |
| elasticnet | 0.20556 | 0.19493 | |

Equal-weight averages of the same four fits, on the same sealed rows, beat
**every** single model — the three-tree average by 0.0164, about seven times what
the whole feature search bought from `t8` onward.

That was [0042](../docs/decisions/0042-the-terminal-argmax-shipped-the-worst-model.md),
which measured the effect and deliberately **did not implement it**, because the
numbers came from v7's own holdout. It registered the claim instead. Kaggle's
private leaderboard has since agreed — blend **0.13813** against the argmax
winner's **0.14356** — and
[0044](../docs/decisions/0044-what-ships-is-the-average-not-the-argmax.md) is the
implementation.

This notebook builds the argument by hand before letting the harness do it:
why picking is expensive, why the *order* of the target inversion is an
estimator choice, and what `finalize_run` now writes.

## 1. Picking is itself a source of error

Forget models for a moment. Take `k` candidates that are **genuinely equally
good** — identical true error of 0.30 — and score each on a noisy validation set.
The scores differ purely by luck. Now take the best one.

You have not picked the best candidate. You have picked the luckiest one, and
luck does not repeat on the next set of rows.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(0)
K, TRIALS = 4, 2000

argmax_gap, average_gap = [], []
for _ in range(TRIALS):
    truth = np.full(K, 0.30)          # all four are equally good, by construction
    val = truth + rng.normal(scale=0.01, size=K)
    sealed = truth + rng.normal(scale=0.01, size=K)
    pick = int(np.argmin(val))        # lower is better
    argmax_gap.append(sealed[pick] - val[pick])
    average_gap.append(sealed.mean() - val.mean())

print(f"argmax  : val flattered it by {np.mean(argmax_gap):+.4f} on average")
print(f"average : val flattered it by {np.mean(average_gap):+.4f} on average")
print(f"\nexpected argmax cost ~ sqrt(2*ln k) SEs = {np.sqrt(2*np.log(K))*0.01:.4f}")

There is no signal anywhere in that simulation — every candidate is identical.
The whole gap is the cost of choosing, and it grows like `sqrt(2*ln k)`.

**This is what `optimism` has been measuring all along.** v7's shipped model came
in +0.0227 worse on sealed rows than on validation. That was never mostly the
model being bad; it was mostly the price of having selected it.

The average pays nothing, because it makes no choice.

## 2. The same thing with real fits

Three disjoint parts, the way the harness cuts them: the models see `work`, the
argmax ranks them on `val`, and `sealed` judges the result exactly once.

In [ ]:
def make_data(n=600, seed=1):
    r = np.random.default_rng(seed)
    x1, x2, x3 = r.normal(size=n), r.normal(size=n), r.normal(size=n)
    y = np.exp(9 + 0.8 * x1 - 0.4 * x2 + 0.3 * x1 * x2 + r.normal(scale=0.25, size=n))
    return pd.DataFrame({"x1": x1, "x2": x2, "x3": x3, "price": y})


data = make_data()
work, val_rows, sealed_rows = data.iloc[:300], data.iloc[300:450], data.iloc[450:]
print(f"work {len(work)} | val {len(val_rows)} | sealed {len(sealed_rows)}")
print(work.head(3).round(3))

In [ ]:
from tabular_agent.cv import make_folds
from tabular_agent.metrics import compute_metric
from tabular_agent.models import predict_model, train_cv

FEATURES = ["x1", "x2", "x3"]
X, y = work[FEATURES], work["price"]
# Fitted on log1p(price), the way the harness fits when the metric is rmsle — so
# the inversion in section 3 is the real one and not a toy.
folds = make_folds(np.log1p(y), "regression", n_splits=3, seed=42)

fits = {}
for model_type in ("lgbm", "xgb", "catboost", "elasticnet"):
    res = train_cv(X, np.log1p(y), folds, "regression", "rmse", model_type=model_type)
    fits[model_type] = res.final_model
print("fitted:", list(fits))

In [ ]:
def score(pred, rows, already_raw=False):
    p = pred if already_raw else np.expm1(pred)
    return compute_metric("rmsle", rows["price"], p)


val_scores, sealed_scores, sealed_preds = {}, {}, {}
for name, m in fits.items():
    val_scores[name] = score(predict_model(m, val_rows[FEATURES], "regression"), val_rows)
    p = np.expm1(predict_model(m, sealed_rows[FEATURES], "regression"))
    sealed_preds[name] = p
    sealed_scores[name] = score(p, sealed_rows, already_raw=True)

table = pd.DataFrame({"val": val_scores, "sealed": sealed_scores}).sort_values("val")
print(table.round(5))

winner = table.index[0]
blend = np.mean(list(sealed_preds.values()), axis=0)
print(f"\nargmax picks '{winner}' -> sealed {sealed_scores[winner]:.5f}")
print(f"equal-weight average    -> sealed {score(blend, sealed_rows, True):.5f}")

## 3. Where the average is taken is an estimator choice

The models predict `log1p(price)`. There are two ways to combine them, and they
are **not the same number**:

- invert each model's prediction to price, then average → an *arithmetic* mean of price
- average on the log scale, then invert once → a *geometric* mean of price

By Jensen's inequality the geometric mean sits strictly below the arithmetic one
wherever the members disagree at all, and they disagree more the more the models
disagree — which is exactly when a blend is doing something.

`predict_bundle` used to do the second one. Every number in 0042, and both Kaggle
submissions, were the first. So the library would have shipped an estimator that
had never been scored against anything, while the record quoted a number
belonging to a different one. It now inverts first.

In [ ]:
each_log = [predict_model(m, sealed_rows[FEATURES], "regression") for m in fits.values()]
arithmetic = np.mean([np.expm1(p) for p in each_log], axis=0)
geometric = np.expm1(np.mean(each_log, axis=0))

print(f"invert then average (what ships) : {score(arithmetic, sealed_rows, True):.5f}")
print(f"average then invert (rejected)   : {score(geometric, sealed_rows, True):.5f}")
print(f"\nthey differ on {(~np.isclose(arithmetic, geometric)).sum()} of {len(arithmetic)} rows")
print(f"largest disagreement: {np.abs(arithmetic - geometric).max():,.0f} in price units")

For a **single-model** bundle the two orders are identical, which is why nothing
already on disk changed: until 0044 every bundle the library wrote had one model
in it.

## 4. Now the harness

Everything above is what `finalize_run` now does on its own. A run with a sealed
third split, so we get the honest number as well.

In [ ]:
import os
import tempfile

from tabular_agent.harness.session import RunSession
from tabular_agent.taskspec import TaskSpec

workdir = tempfile.mkdtemp()
csv = os.path.join(workdir, "prices.csv")
make_data(n=900, seed=3).to_csv(csv, index=False)

session = RunSession(
    os.path.join(workdir, "run"),
    csv,
    TaskSpec(target="price", metric="rmsle", n_splits=3),
    holdout_fraction=0.2,
)
# rmsle implies log1p, derived from the metric rather than chosen (0035).
print(session.resolved().target_transform, "|", session.resolved().metric)

In [ ]:
from tabular_agent.harness.tools import finalize_run, inspect_data, transform_features

inspect_data(session)
transform_features(
    session,
    "def transform_features(df):\n"
    "    df = df.copy()\n"
    "    df['x1x2'] = df['x1'] * df['x2']\n"
    "    return df",
)
out = finalize_run(session)

print("model_type   :", out["model_type"])
print("members      :", out["blend_members"])
print("val_score    :", out["val_score"])
print("holdout_score:", out["holdout_score"])
print("optimism     :", out["optimism"])

The bundle holds every fit at equal weight. Nothing was tuned and nothing was
dropped — a filter would be a selection, which is the cost we are refusing to
pay, and the obvious filter ("keep the boosted trees") is refuted by house v9,
where `elasticnet` was the best model on the board.

In [ ]:
from tabular_agent.pipeline import load_bundle, predict_bundle

bundle = load_bundle(os.path.join(session.run_dir, out["bundle_dir"]))
print("models in the bundle:", len(bundle["models"]))
print("weights             :", bundle["weights"])
print("target_transform    :", bundle["target_transform"])

fresh = make_data(n=5, seed=99).drop(columns=["price"])
print("\npredictions on raw rows:", predict_bundle(bundle, fresh).round(0))

## 5. The A/B, recomputed by every run

Two flags, and they mean different things:

- **`selected`** — the candidate the *search* crowned. Kept as its own field
  because the optimism of the selected candidate is the series 0039 and 0040 have
  been tracking across runs, and folding it into "what shipped" would break that
  series at the run where the blend arrived.
- **`shipped`** — what the bundle actually predicts with.

Their difference is 0042's A/B, and every run now produces one. That is the point:
the decision stays falsifiable instead of being settled by the run that made it.

In [ ]:
from tabular_agent.harness import journal

scores = journal.load(session.run_dir)["holdout"]["scores"]
print(pd.DataFrame(scores)[
    ["experiment_id", "model_type", "val_score", "holdout_score", "optimism",
     "selected", "shipped"]
].round(5).to_string(index=False))

In [ ]:
report = open(os.path.join(session.run_dir, out["report_path"])).read()
start = report.index("## What the search cost")
print(report[start:start + 1400])

**Do not expect the blend to win every time, and this notebook probably shows it
losing.** On 900 synthetic rows whose truth is nearly log-linear, `elasticnet`
is close to the right model and averaging it with three boosted trees makes it
worse. That is a real failure mode, not a flaw in the demo — it is house v9's
situation, and it is exactly why the report prints this comparison on every run
rather than asserting the answer once.

What the run cannot do is *use* that number. The blend's members are fixed before
any sealed row is read: `_blend_members` runs off the terminal fits, and
`_score_on_holdout` receives the finished blend as an argument, the same way it
receives the finished winner. The seal judges; it never picks (0023 rule 2).

## What this cost

- **What ships was never cross-validated.** A blend has no folds of its own, so
  the report prints no CV score, no fold spread and no per-fold table for it.
- **The bundle is `k` times larger and slower.** Four boosted-tree models, not one.
- **Attribution still runs off the shipped table's lineage** — every member shares
  it — but "which columns earned the score" now describes a score no single
  fitted model produced.

## Registered for the next sealed run

1. **The blend's optimism is smaller than the selected candidate's.** The
   mechanism from section 1, stated as a prediction.
2. **The blend beats the argmax on sealed rows, on a dataset that is not
   Rossmann.** The generalisation Kaggle cannot supply.
3. **0039's pattern holds a fourth time** — the selected candidate is the most
   optimistic row on the ballot — now with a shipped model that is not on it.